In [1]:
import pandas as pd
import os

# Load golden dataset
golden_folder = "golden_dataset"

data = {}

for file in os.listdir(golden_folder):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        data[name] = pd.read_csv(os.path.join(golden_folder, file))

print("Golden dataset loaded successfully.")
print("Total tables:", len(data))

Golden dataset loaded successfully.
Total tables: 18


In [2]:
# Check duplicate payments

payments = data["payments"]

duplicate_payments = payments[payments.duplicated(keep=False)]

print("Total duplicate rows:", len(duplicate_payments))
print("Unique duplicate payment IDs:",
      duplicate_payments["payment_id"].nunique())

duplicate_payments.head(10)


Total duplicate rows: 0
Unique duplicate payment IDs: 0


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id


In [3]:
# Load raw payments data

raw_payments = pd.read_csv("payments.csv")

duplicate_payments = raw_payments[
    raw_payments.duplicated(keep=False)
]

print("Total duplicate rows:", len(duplicate_payments))
print("Unique duplicate payment IDs:",
      duplicate_payments["payment_id"].nunique())

duplicate_payments.head(10)

Total duplicate rows: 972
Unique duplicate payment IDs: 486


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
55,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
75,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
107,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
148,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
198,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012
254,PAYMENT0000255,ACC0001307,BRW0001928,2026-05-07 18:37:15,TXN0000016416,5970.90,PENDING,UPI,VND0000005
262,PAYMENT0000263,ACC0004813,BRW0004424,2026-02-03 14:14:28,TXN0000019152,25143.15,PENDING,CARD,VND0000014
310,PAYMENT0000311,ACC0010109,BRW0005134,2026-04-09 20:18:27,TXN0000013702,75456.24,REVERSED,NETBANKING,VND0000012
551,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
552,PAYMENT0000553,ACC0026650,BRW0004833,2026-05-01 05:10:02,TXN0000001233,45817.54,REVERSED,NACH,VND0000002


In [4]:
# Calculate duplicate payment impact

duplicate_rows = raw_payments[
    raw_payments.duplicated(keep="first")
]

duplicate_amount = duplicate_rows["amount"].sum()
total_amount = raw_payments["amount"].sum()

print("Total payment amount:", round(total_amount, 2))
print("Duplicate amount:", round(duplicate_amount, 2))
print("Amount inflated by duplicates:",
      round((duplicate_amount / total_amount) * 100, 2), "%")

Total payment amount: 1917258617.15
Duplicate amount: 37299019.8
Amount inflated by duplicates: 1.95 %


In [5]:
# Check duplicate impact by status

duplicate_status = (
    duplicate_rows
    .groupby("payment_status")
    .agg(
        duplicate_rows=("payment_id", "count"),
        duplicate_amount=("amount", "sum")
    )
    .sort_values("duplicate_amount", ascending=False)
)

duplicate_status

,duplicate_rows,duplicate_amount
payment_status,,
SUCCESS,335,25011462.19
FAILED,66,5060989.58
PENDING,56,4502633.24
REVERSED,29,2723934.79


In [6]:
# Check payment attribution

payments = raw_payments.copy()

print("Payment columns:")
print(payments.columns.tolist())

Payment columns:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']


In [7]:
# Check related tables

print("Calls columns:")
print(data["calls"].columns.tolist())

print("\nCampaign columns:")
print(data["campaigns"].columns.tolist())

print("\nAccounts columns:")
print(data["accounts"].columns.tolist())

Calls columns:
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']

Campaign columns:
['campaign_id', 'campaign_name', 'channel', 'strategy_version', 'start_at', 'target_definition', 'end_at']

Accounts columns:
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']


In [8]:
# Find latest call before each payment

calls = data["calls"].copy()

calls["event_at"] = pd.to_datetime(calls["event_at"])
payments["event_at"] = pd.to_datetime(payments["event_at"])

calls = calls.sort_values("event_at")
payments = payments.sort_values("event_at")

payment_calls = pd.merge_asof(
    payments,
    calls[["account_id", "event_at", "campaign_id"]],
    on="event_at",
    by="account_id",
    direction="backward",
    suffixes=("_payment", "_call")
)

payment_calls[[
    "payment_id",
    "account_id",
    "event_at",
    "campaign_id"
]].head(10)

,payment_id,account_id,event_at,campaign_id
0,PAYMENT0008675,ACC0014142,2026-01-01 00:14:40,NaN
1,PAYMENT0008644,ACC0003318,2026-01-01 00:27:39,NaN
2,PAYMENT0002693,ACC0014449,2026-01-01 00:41:58,NaN
3,PAYMENT0004607,ACC0018164,2026-01-01 00:44:12,NaN
4,PAYMENT0000307,ACC0026311,2026-01-01 00:46:12,NaN
5,PAYMENT0024821,ACC0018361,2026-01-01 00:50:36,NaN
6,PAYMENT0011191,ACC0004322,2026-01-01 01:02:22,NaN
7,PAYMENT0008443,ACC0005569,2026-01-01 01:21:07,NaN
8,PAYMENT0020533,ACC0002164,2026-01-01 01:21:45,NaN
9,PAYMENT0018250,ACC0022477,2026-01-01 01:33:19,NaN


In [9]:
# Check attribution coverage

matched = payment_calls["campaign_id"].notna()

print("Total payments:", len(payment_calls))
print("Payments with previous call:", matched.sum())
print("Payments without previous call:", (~matched).sum())

print("\nAttribution coverage:")
print(round(matched.mean() * 100, 2), "%")

Total payments: 25500
Payments with previous call: 17356
Payments without previous call: 8144

Attribution coverage:
68.06 %


In [10]:
print(payment_calls.columns.tolist())

['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id', 'campaign_id']


In [11]:
# Check payment attribution

payments = data["payments"].copy()
calls = data["calls"].copy()

payments["event_at"] = pd.to_datetime(payments["event_at"])
calls["event_at"] = pd.to_datetime(calls["event_at"])

payments = payments.sort_values("event_at")
calls = calls.sort_values("event_at")

attribution = pd.merge_asof(
    payments,
    calls[["account_id", "event_at", "campaign_id"]],
    on="event_at",
    by="account_id",
    direction="backward",
    suffixes=("_payment", "_call")
)

print("Total payments:", len(attribution))
print("Attributed payments:", attribution["campaign_id"].notna().sum())
print("Unattributed payments:", attribution["campaign_id"].isna().sum())

attribution.head(10)

Total payments: 25014
Attributed payments: 17025
Unattributed payments: 7989


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id,campaign_id
0,PAYMENT0008675,ACC0014142,BRW0001216,2026-01-01 00:14:40,TXN0000000959,99867.36,SUCCESS,CARD,VND0000009,NaN
1,PAYMENT0008644,ACC0003318,BRW0008843,2026-01-01 00:27:39,TXN0000062945,94128.59,REVERSED,CASH,VND0000005,NaN
2,PAYMENT0002693,ACC0014449,BRW0001005,2026-01-01 00:41:58,TXN0000042163,101870.25,SUCCESS,CARD,VND0000009,NaN
3,PAYMENT0004607,ACC0018164,BRW0008851,2026-01-01 00:44:12,TXN0000022605,63882.71,FAILED,UPI,VND0000009,NaN
4,PAYMENT0000307,ACC0026311,BRW0004699,2026-01-01 00:46:12,TXN0000047736,68439.57,FAILED,CASH,VND0000007,NaN
5,PAYMENT0024821,ACC0018361,BRW0008646,2026-01-01 00:50:36,TXN0000028353,87906.80,SUCCESS,NACH,VND0000002,NaN
6,PAYMENT0011191,ACC0004322,BRW0006412,2026-01-01 01:02:22,TXN0000051585,94150.20,SUCCESS,CASH,VND0000009,NaN
7,PAYMENT0008443,ACC0005569,BRW0009161,2026-01-01 01:21:07,TXN0000040452,81954.29,SUCCESS,CARD,VND0000010,NaN
8,PAYMENT0020533,ACC0002164,BRW0005627,2026-01-01 01:21:45,TXN0000046927,74524.80,SUCCESS,CASH,VND0000006,NaN
9,PAYMENT0018250,ACC0022477,BRW0004233,2026-01-01 01:33:19,TXN0000014551,23224.76,PENDING,CARD,VND0000010,NaN


In [12]:
# Check attribution by campaign

print("Attributed payments by campaign:")
print(attribution["campaign_id"].value_counts().head(10))

Attributed payments by campaign:
campaign_id
CMP0000109    175
CMP0000092    173
CMP0000005    170
CMP0000116    167
CMP0000008    166
CMP0000055    165
CMP0000098    165
CMP0000115    165
CMP0000060    165
CMP0000036    163
Name: count, dtype: int64


In [13]:
# Check attribution timing

attribution["event_at"] = pd.to_datetime(attribution["event_at"])

print("Attribution columns:")
print(attribution.columns.tolist())

Attribution columns:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id', 'campaign_id']


In [14]:
# Build attribution with call time

payments = data["payments"].copy()
calls = data["calls"].copy()

payments["payment_time"] = pd.to_datetime(payments["event_at"])
calls["call_time"] = pd.to_datetime(calls["event_at"])

payments = payments.sort_values("payment_time")
calls = calls.sort_values("call_time")

attribution = pd.merge_asof(
    payments,
    calls[["account_id", "call_time", "campaign_id"]],
    left_on="payment_time",
    right_on="call_time",
    by="account_id",
    direction="backward"
)

print("Total payments:", len(attribution))
print("Attributed payments:", attribution["campaign_id"].notna().sum())
print("Unattributed payments:", attribution["campaign_id"].isna().sum())

attribution.head(10)

Total payments: 25014
Attributed payments: 17025
Unattributed payments: 7989


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id,payment_time,call_time,campaign_id
0,PAYMENT0008675,ACC0014142,BRW0001216,2026-01-01 00:14:40,TXN0000000959,99867.36,SUCCESS,CARD,VND0000009,2026-01-01 00:14:40,NaT,NaN
1,PAYMENT0008644,ACC0003318,BRW0008843,2026-01-01 00:27:39,TXN0000062945,94128.59,REVERSED,CASH,VND0000005,2026-01-01 00:27:39,NaT,NaN
2,PAYMENT0002693,ACC0014449,BRW0001005,2026-01-01 00:41:58,TXN0000042163,101870.25,SUCCESS,CARD,VND0000009,2026-01-01 00:41:58,NaT,NaN
3,PAYMENT0004607,ACC0018164,BRW0008851,2026-01-01 00:44:12,TXN0000022605,63882.71,FAILED,UPI,VND0000009,2026-01-01 00:44:12,NaT,NaN
4,PAYMENT0000307,ACC0026311,BRW0004699,2026-01-01 00:46:12,TXN0000047736,68439.57,FAILED,CASH,VND0000007,2026-01-01 00:46:12,NaT,NaN
5,PAYMENT0024821,ACC0018361,BRW0008646,2026-01-01 00:50:36,TXN0000028353,87906.80,SUCCESS,NACH,VND0000002,2026-01-01 00:50:36,NaT,NaN
6,PAYMENT0011191,ACC0004322,BRW0006412,2026-01-01 01:02:22,TXN0000051585,94150.20,SUCCESS,CASH,VND0000009,2026-01-01 01:02:22,NaT,NaN
7,PAYMENT0008443,ACC0005569,BRW0009161,2026-01-01 01:21:07,TXN0000040452,81954.29,SUCCESS,CARD,VND0000010,2026-01-01 01:21:07,NaT,NaN
8,PAYMENT0020533,ACC0002164,BRW0005627,2026-01-01 01:21:45,TXN0000046927,74524.80,SUCCESS,CASH,VND0000006,2026-01-01 01:21:45,NaT,NaN
9,PAYMENT0018250,ACC0022477,BRW0004233,2026-01-01 01:33:19,TXN0000014551,23224.76,PENDING,CARD,VND0000010,2026-01-01 01:33:19,NaT,NaN


In [15]:
# Check attribution timing

attribution["payment_time"] = pd.to_datetime(attribution["payment_time"])
attribution["call_time"] = pd.to_datetime(attribution["call_time"])

attribution["lag_hours"] = (
    attribution["payment_time"] - attribution["call_time"]
).dt.total_seconds() / 3600

print("Average attribution lag:", round(attribution["lag_hours"].mean(), 2), "hours")
print("Median attribution lag:", round(attribution["lag_hours"].median(), 2), "hours")
print("Maximum attribution lag:", round(attribution["lag_hours"].max(), 2), "hours")

print("\nLongest attribution lags:")
print(
    attribution[
        attribution["campaign_id"].notna()
    ][["payment_id", "account_id", "call_time", "payment_time", "campaign_id", "lag_hours"]]
    .sort_values("lag_hours", ascending=False)
    .head(10)
)

Average attribution lag: 1061.48 hours
Median attribution lag: 801.01 hours
Maximum attribution lag: 5117.72 hours

Longest attribution lags:
           payment_id  account_id           call_time        payment_time  \
24626  PAYMENT0011041  ACC0017783 2026-01-04 09:54:32 2026-08-05 15:37:45   
24988  PAYMENT0021809  ACC0022568 2026-01-09 16:12:59 2026-08-08 18:53:12   
24810  PAYMENT0017308  ACC0024429 2026-01-08 20:54:35 2026-08-07 05:02:19   
24956  PAYMENT0022124  ACC0007141 2026-01-11 16:37:05 2026-08-08 11:47:30   
23825  PAYMENT0019860  ACC0013141 2026-01-04 00:15:18 2026-07-29 06:38:30   
23959  PAYMENT0007924  ACC0002194 2026-01-05 13:58:15 2026-07-30 15:14:44   
23780  PAYMENT0002536  ACC0003407 2026-01-04 05:36:44 2026-07-28 21:28:58   
23801  PAYMENT0022754  ACC0027738 2026-01-04 15:10:07 2026-07-29 01:11:21   
24756  PAYMENT0010744  ACC0016499 2026-01-13 22:43:25 2026-08-06 19:33:48   
24297  PAYMENT0016460  ACC0001543 2026-01-10 20:20:26 2026-08-02 15:54:42   

      camp

In [16]:
# Check attribution window

for days in [7, 14, 30, 60]:
    valid = attribution["lag_hours"].between(0, days * 24)
    print(f"Within {days} days:", valid.sum(), 
          f"({round(valid.mean() * 100, 2)}%)")
    

Within 7 days: 2237 (8.94%)
Within 14 days: 4163 (16.64%)
Within 30 days: 7868 (31.45%)
Within 60 days: 12425 (49.67%)


In [17]:
# Check agent identity

agents = data["agents"]

print("Total agents:", len(agents))
print("Unique agent IDs:", agents["agent_id"].nunique())

print("\nAgent names with multiple IDs:")
agent_names = agents.groupby("agent_name")["agent_id"].nunique()
print(agent_names[agent_names > 1])

Total agents: 30000
Unique agent IDs: 1000

Agent names with multiple IDs:
agent_name
Aarav Sharma    946
Amit Kumar      952
Ananya Rao      946
Neha Singh      951
Pooja Nair      954
Priya Mehta     957
Rahul Verma     943
Rohan Patel     946
Sneha Das       958
Vikram Shah     936
Name: agent_id, dtype: int64


In [18]:
# Check agent ID reuse

id_counts = agents["agent_id"].value_counts()

print("IDs used once:", (id_counts == 1).sum())
print("IDs used multiple times:", (id_counts > 1).sum())

print("\nMost repeated agent IDs:")
print(id_counts.head(10))

IDs used once: 0
IDs used multiple times: 1000

Most repeated agent IDs:
agent_id
AGT0000367    48
AGT0000533    48
AGT0000875    48
AGT0000843    47
AGT0000540    46
AGT0000876    46
AGT0000563    45
AGT0000543    45
AGT0000440    45
AGT0000975    44
Name: count, dtype: int64


In [19]:
# Check portfolio mix over time

accounts = data["accounts"].copy()
accounts["opened_at"] = pd.to_datetime(accounts["opened_at"])

accounts["month"] = accounts["opened_at"].dt.to_period("M")

mix = pd.crosstab(
    accounts["month"],
    accounts["loan_type"],
    normalize="index"
) * 100

print("Loan type mix by month:")
print(mix.round(2))

Loan type mix by month:
loan_type   AUTO   BNPL  CONSUMER  CREDIT_CARD  PERSONAL
month                                                   
2024-01    20.59  19.07     20.14        19.98     20.21
2024-02    21.40  19.17     19.71        18.63     21.09
2024-03    19.02  22.86     19.66        19.94     18.52
2024-04    19.95  18.69     21.22        21.30     18.84
2024-05    19.97  20.20     18.41        21.52     19.89
2024-06    19.44  21.66     17.88        19.93     21.08
2024-07    20.54  22.13     16.84        19.34     21.15
2024-08    19.59  18.34     20.54        20.84     20.69
2024-09    21.95  20.21     17.50        19.68     20.66
2024-10    17.84  19.46     20.48        22.91     19.31
2024-11    21.20  18.87     20.57        18.87     20.50
2024-12    20.37  18.68     21.82        20.52     18.61
2025-01    20.20  18.39     21.40        22.22     17.79
2025-02    22.16  21.35     18.59        18.91     18.99
2025-03    19.19  18.97     20.66        20.66     20.52
2025-04

In [20]:
# Check risk mix over time

risk_mix = pd.crosstab(
    accounts["month"],
    accounts["risk_segment"],
    normalize="index"
) * 100

print("Risk segment mix by month:")
print(risk_mix.round(2))

Risk segment mix by month:
risk_segment   HIGH    LOW  MEDIUM    NPA
month                                    
2024-01       24.18  26.09   26.32  23.42
2024-02       24.17  25.17   25.71  24.94
2024-03       26.85  23.79   24.86  24.50
2024-04       26.37  26.68   23.59  23.36
2024-05       26.65  25.17   24.32  23.85
2024-06       26.25  25.18   25.35  23.22
2024-07       26.59  25.00   24.02  24.40
2024-08       24.23  23.27   26.73  25.77
2024-09       26.17  23.76   25.04  25.04
2024-10       26.36  24.16   27.09  22.39
2024-11       24.69  25.62   24.30  25.39
2024-12       24.43  24.81   24.35  26.42
2025-01       24.85  25.90   26.13  23.12
2025-02       25.97  25.49   23.78  24.76
2025-03       26.05  24.06   24.50  25.39
2025-04       23.69  25.04   23.85  27.42
2025-05       25.61  25.61   23.17  25.61
2025-06       23.33  25.24   27.07  24.36
2025-07       24.45  24.45   25.93  25.16
2025-08       25.02  25.47   25.47  24.04
2025-09       24.25  24.25   26.30  25.20
2025-10

In [23]:
# Check account status mix

status_counts = accounts["status"].value_counts()

print("Account status counts:")
print(status_counts)

print("\nAccount status percentage:")
print((status_counts / len(accounts) * 100).round(2))

Account status counts:
status
ACTIVE      7539
CLOSED      7496
PAID        7486
WRITEOFF    7479
Name: count, dtype: int64

Account status percentage:
status
ACTIVE      25.13
CLOSED      24.99
PAID        24.95
WRITEOFF    24.93
Name: count, dtype: float64


In [24]:

print(history.columns.tolist())

['history_id', 'account_id', 'borrower_id', 'event_at', 'status', 'changed_by', 'source', 'recorded_at']


In [25]:
# Check status changes over time

history = data["account_status_history"].copy()

history["event_at"] = pd.to_datetime(history["event_at"])

status_history = pd.crosstab(
    history["event_at"].dt.to_period("M"),
    history["status"]
)

print("Monthly account status changes:")
print(status_history)

Monthly account status changes:
status    ACTIVE  CLOSED  DELINQUENT   NPA  PAID   PTP  WRITEOFF
event_at                                                        
2026-01     1184    1220        1236  1189  1249  1182      1170
2026-02     1092    1074        1093  1102  1096  1066      1069
2026-03     1232    1206        1276  1262  1210  1135      1208
2026-04     1121    1210        1174  1122  1136  1123      1231
2026-05     1200    1201        1217  1156  1256  1243      1213
2026-06     1170    1165        1145  1210  1180  1182      1189
2026-07     1204    1240        1189  1257  1212  1172      1201
2026-08      315     298         282   314   311   308       302


In [26]:
# Check monthly status percentage

status_percentage = status_history.div(
    status_history.sum(axis=1),
    axis=0
) * 100

print("Monthly account status percentage:")
print(status_percentage.round(2))

Monthly account status percentage:
status    ACTIVE  CLOSED  DELINQUENT    NPA   PAID    PTP  WRITEOFF
event_at                                                           
2026-01    14.05   14.47       14.66  14.10  14.82  14.02     13.88
2026-02    14.38   14.15       14.40  14.52  14.44  14.04     14.08
2026-03    14.44   14.14       14.96  14.80  14.19  13.31     14.16
2026-04    13.81   14.91       14.46  13.82  14.00  13.84     15.17
2026-05    14.14   14.15       14.34  13.62  14.80  14.65     14.29
2026-06    14.20   14.14       13.89  14.68  14.32  14.34     14.43
2026-07    14.21   14.63       14.03  14.83  14.30  13.83     14.17
2026-08    14.79   13.99       13.24  14.74  14.60  14.46     14.18


In [27]:
# Check status transitions

history = history.sort_values(["account_id", "event_at"])

history["previous_status"] = (
    history.groupby("account_id")["status"].shift(1)
)

transitions = history.dropna(subset=["previous_status"]).copy()

transition_counts = pd.crosstab(
    transitions["previous_status"],
    transitions["status"]
)

print("Status transition counts:")
print(transition_counts)

Status transition counts:
status           ACTIVE  CLOSED  DELINQUENT  NPA  PAID  PTP  WRITEOFF
previous_status                                                      
ACTIVE              699     711         704  675   666  710       663
CLOSED              685     710         725  654   732  667       718
DELINQUENT          681     729         720  722   718  696       705
NPA                 683     690         645  710   735  713       687
PAID                696     698         707  705   697  681       706
PTP                 637     618         665  694   661  722       707
WRITEOFF            704     658         729  707   716  661       679


In [28]:
# Check unusual status transitions

unusual_transitions = transitions[
    transitions["previous_status"] != transitions["status"]
].copy()

print("Total status transitions:", len(unusual_transitions))

print("\nMost common status transitions:")
print(
    unusual_transitions
    .groupby(["previous_status", "status"])
    .size()
    .sort_values(ascending=False)
    .head(15)
)

Total status transitions: 29064

Most common status transitions:
previous_status  status    
NPA              PAID          735
CLOSED           PAID          732
WRITEOFF         DELINQUENT    729
DELINQUENT       CLOSED        729
CLOSED           DELINQUENT    725
DELINQUENT       NPA           722
CLOSED           WRITEOFF      718
DELINQUENT       PAID          718
WRITEOFF         PAID          716
NPA              PTP           713
ACTIVE           CLOSED        711
                 PTP           710
PAID             DELINQUENT    707
WRITEOFF         NPA           707
PTP              WRITEOFF      707
dtype: int64


In [29]:
# Check number of status changes per account

status_change_counts = (
    unusual_transitions
    .groupby("account_id")
    .size()
    .sort_values(ascending=False)
)

print("Accounts with most status changes:")
print(status_change_counts.head(15))

print("\nAverage status changes per account:")
print(round(status_change_counts.mean(), 2))

Accounts with most status changes:
account_id
ACC0008596    8
ACC0024595    7
ACC0001116    7
ACC0003466    7
ACC0012736    7
ACC0025503    7
ACC0006825    7
ACC0003706    7
ACC0006925    7
ACC0009758    7
ACC0012977    7
ACC0015214    7
ACC0011891    6
ACC0011638    6
ACC0029379    6
dtype: int64

Average status changes per account:
1.76


In [30]:
# Check duplicate status history records

history_duplicates = history.duplicated().sum()

print("Duplicate status history rows:", history_duplicates)

print("\nDuplicate history IDs:", history["history_id"].duplicated().sum())

print("\nUnique history IDs:", history["history_id"].nunique())
print("Total history rows:", len(history))

Duplicate status history rows: 0

Duplicate history IDs: 0

Unique history IDs: 60000
Total history rows: 60000


In [31]:
# Check multiple status records at the same timestamp

same_timestamp = (
    history
    .groupby(["account_id", "event_at"])
    .size()
    .sort_values(ascending=False)
)

print("Maximum records for same account and timestamp:")
print(same_timestamp.max())

print("\nAccount-timestamp combinations with multiple records:")
print(same_timestamp[same_timestamp > 1].head(15))

Maximum records for same account and timestamp:
1

Account-timestamp combinations with multiple records:
Series([], dtype: int64)


In [32]:
# Check rapid status changes

history = history.sort_values(["account_id", "event_at"])

history["previous_event_at"] = (
    history.groupby("account_id")["event_at"].shift(1)
)

history["hours_since_previous"] = (
    history["event_at"] - history["previous_event_at"]
).dt.total_seconds() / 3600

rapid_changes = history[
    (history["hours_since_previous"] >= 0) &
    (history["hours_since_previous"] < 24) &
    (history["previous_status"] != history["status"])
].copy()

print("Status changes within 24 hours:", len(rapid_changes))

print("\nExamples of rapid status changes:")
print(
    rapid_changes[
        ["account_id", "previous_status", "status",
         "previous_event_at", "event_at", "hours_since_previous"]
    ].head(15)
)


Status changes within 24 hours: 481

Examples of rapid status changes:
       account_id previous_status      status   previous_event_at  \
35304  ACC0000075          ACTIVE      CLOSED 2026-01-31 02:44:31   
40034  ACC0000122             PTP        PAID 2026-01-04 19:30:58   
6875   ACC0000248             NPA        PAID 2026-04-25 18:44:10   
8285   ACC0000399             NPA  DELINQUENT 2026-04-06 20:10:09   
26255  ACC0000407             NPA    WRITEOFF 2026-04-26 17:18:45   
48552  ACC0000540             NPA    WRITEOFF 2026-06-05 16:13:25   
19986  ACC0000541            PAID      CLOSED 2026-04-21 18:39:27   
17301  ACC0000597          ACTIVE  DELINQUENT 2026-02-16 13:58:46   
26067  ACC0000766        WRITEOFF  DELINQUENT 2026-04-20 23:58:41   
51179  ACC0000834            PAID         NPA 2026-05-21 18:31:40   
16956  ACC0000980          ACTIVE         NPA 2026-02-02 02:30:05   
21489  ACC0001001            PAID      CLOSED 2026-06-18 05:17:30   
20273  ACC0001042          CLOSE

In [33]:
# Check status changes within 1 hour

very_rapid = history[
    (history["hours_since_previous"] >= 0) &
    (history["hours_since_previous"] < 1) &
    (history["previous_status"] != history["status"])
].copy()

print("Status changes within 1 hour:", len(very_rapid))

print("\nFastest status changes:")
print(
    very_rapid[
        ["account_id", "previous_status", "status",
         "previous_event_at", "event_at", "hours_since_previous"]
    ]
    .sort_values("hours_since_previous")
    .head(15)
)

Status changes within 1 hour: 15

Fastest status changes:
       account_id previous_status      status   previous_event_at  \
39718  ACC0029009             PTP  DELINQUENT 2026-01-28 12:47:06   
38547  ACC0020030            PAID         PTP 2026-07-29 11:18:17   
12839  ACC0029959          CLOSED         PTP 2026-05-13 17:33:56   
52242  ACC0017634             PTP        PAID 2026-06-20 05:26:41   
32976  ACC0009875          ACTIVE      CLOSED 2026-05-11 17:32:36   
35304  ACC0000075          ACTIVE      CLOSED 2026-01-31 02:44:31   
38330  ACC0020586      DELINQUENT    WRITEOFF 2026-07-26 08:36:50   
15080  ACC0027958             NPA        PAID 2026-05-14 09:14:56   
21675  ACC0018788             PTP      CLOSED 2026-08-07 14:14:13   
58417  ACC0014697      DELINQUENT      ACTIVE 2026-05-01 14:28:24   
57331  ACC0009191        WRITEOFF         NPA 2026-06-13 18:19:42   
43102  ACC0021938            PAID      ACTIVE 2026-07-24 04:35:09   
26067  ACC0000766        WRITEOFF  DELINQUENT

In [35]:
# Show examples of status reversals

print("Examples of status reversals:")

print(
    reversals[
        ["account_id", "previous_status", "status"]
    ].head(15)
)

Examples of status reversals:
       account_id previous_status      status
59093  ACC0000008          ACTIVE      CLOSED
39843  ACC0000015      DELINQUENT        PAID
7848   ACC0000027      DELINQUENT        PAID
35304  ACC0000075          ACTIVE      CLOSED
52771  ACC0000091      DELINQUENT        PAID
28966  ACC0000092          CLOSED      ACTIVE
44517  ACC0000094      DELINQUENT        PAID
42167  ACC0000098          ACTIVE      CLOSED
53402  ACC0000099          ACTIVE      CLOSED
35045  ACC0000111          CLOSED      ACTIVE
32757  ACC0000114          CLOSED      ACTIVE
6026   ACC0000131      DELINQUENT        PAID
44830  ACC0000132            PAID  DELINQUENT
6423   ACC0000152          CLOSED      ACTIVE
9522   ACC0000169      DELINQUENT        PAID


In [36]:
# Check status changes by source

print("Status changes by source:")

print(
    history["source"]
    .value_counts()
)

print("\nStatus changes by source and status:")

print(
    pd.crosstab(
        history["source"],
        history["status"]
    )
)

Status changes by source:
source
CORE       12079
BATCH      12073
CALL       12028
FIELD      11911
PAYMENT    11909
Name: count, dtype: int64

Status changes by source and status:
status   ACTIVE  CLOSED  DELINQUENT   NPA  PAID   PTP  WRITEOFF
source                                                         
BATCH      1678    1747        1801  1712  1729  1702      1704
CALL       1704    1715        1711  1760  1769  1636      1733
CORE       1719    1770        1680  1707  1718  1743      1742
FIELD      1708    1692        1744  1719  1741  1642      1665
PAYMENT    1709    1690        1676  1714  1693  1688      1739


In [37]:
# Check who is making status changes

print("Status changes by changed_by:")

print(history["changed_by"].value_counts())

print("\nStatus changes by changed_by and status:")

print(
    pd.crosstab(
        history["changed_by"],
        history["status"]
    )
)

Status changes by changed_by:
changed_by
AGT0000047    644
AGT0000088    637
AGT0000035    635
AGT0000001    633
AGT0000055    632
             ... 
AGT0000020    546
AGT0000007    544
AGT0000015    542
AGT0000064    537
AGT0000049    527
Name: count, Length: 101, dtype: int64

Status changes by changed_by and status:
status      ACTIVE  CLOSED  DELINQUENT  NPA  PAID  PTP  WRITEOFF
changed_by                                                      
AGT0000001      99      78          97   77    83   94       105
AGT0000002      84      89          76   80    95   76        77
AGT0000003      93      97          87   87    72   84       102
AGT0000004     105      91         107   85    82   73        88
AGT0000005      75      73          94   82   112   81        88
...            ...     ...         ...  ...   ...  ...       ...
AGT0000097      89      64          89   83    84   80        93
AGT0000098      95      87          84   68    94  108        95
AGT0000099      78     100    

In [38]:
# Check agent activity outliers

agent_activity = history["changed_by"].value_counts()

print("Average changes per agent:", round(agent_activity.mean(), 2))
print("Maximum changes by one agent:", agent_activity.max())
print("Minimum changes by one agent:", agent_activity.min())

print("\nTop 10 most active agents:")
print(agent_activity.head(10))

print("\nBottom 10 least active agents:")
print(agent_activity.tail(10))

Average changes per agent: 594.06
Maximum changes by one agent: 644
Minimum changes by one agent: 527

Top 10 most active agents:
changed_by
AGT0000047    644
AGT0000088    637
AGT0000035    635
AGT0000001    633
AGT0000055    632
AGT0000012    632
AGT0000084    631
AGT0000098    631
AGT0000066    631
AGT0000004    631
Name: count, dtype: int64

Bottom 10 least active agents:
changed_by
AGT0000090    561
AGT0000033    560
AGT0000016    560
AGT0000100    555
AGT0000087    550
AGT0000020    546
AGT0000007    544
AGT0000015    542
AGT0000064    537
AGT0000049    527
Name: count, dtype: int64


In [39]:
# Check agent activity by source

agent_source = pd.crosstab(
    history["changed_by"],
    history["source"]
)

print("Agent activity by source:")
print(agent_source)

print("\nHighest agent-source combinations:")

print(
    agent_source
    .stack()
    .sort_values(ascending=False)
    .head(15)
)

Agent activity by source:
source      BATCH  CALL  CORE  FIELD  PAYMENT
changed_by                                   
AGT0000001    120   130   134    126      123
AGT0000002    106   138   110     98      125
AGT0000003    134   135   116    120      117
AGT0000004    128   147   115    119      122
AGT0000005    137   111   122    107      128
...           ...   ...   ...    ...      ...
AGT0000097    118   103   136     95      130
AGT0000098    127   117   134    131      122
AGT0000099    110   122   109    126      123
AGT0000100    112   108   118    109      108
SYSTEM        101   134   122    127      115

[101 rows x 5 columns]

Highest agent-source combinations:
changed_by  source
AGT0000027  BATCH     147
AGT0000004  CALL      147
AGT0000058  FIELD     147
AGT0000009  CORE      144
AGT0000084  BATCH     143
AGT0000032  BATCH     143
AGT0000017  FIELD     142
AGT0000037  CORE      141
AGT0000012  CALL      141
AGT0000052  BATCH     141
AGT0000080  CORE      140
AGT0000059 

In [40]:
# Check for agents with unusually high source concentration

agent_source_pct = agent_source.div(
    agent_source.sum(axis=1),
    axis=0
) * 100

print("Agent-source percentage distribution:")
print(agent_source_pct.round(2))

print("\nHighest source concentration by agent:")

max_concentration = agent_source_pct.max(axis=1).sort_values(ascending=False)

print(max_concentration.head(15).round(2))

Agent-source percentage distribution:
source      BATCH   CALL   CORE  FIELD  PAYMENT
changed_by                                     
AGT0000001  18.96  20.54  21.17  19.91    19.43
AGT0000002  18.37  23.92  19.06  16.98    21.66
AGT0000003  21.54  21.70  18.65  19.29    18.81
AGT0000004  20.29  23.30  18.23  18.86    19.33
AGT0000005  22.64  18.35  20.17  17.69    21.16
...           ...    ...    ...    ...      ...
AGT0000097  20.27  17.70  23.37  16.32    22.34
AGT0000098  20.13  18.54  21.24  20.76    19.33
AGT0000099  18.64  20.68  18.47  21.36    20.85
AGT0000100  20.18  19.46  21.26  19.64    19.46
SYSTEM      16.86  22.37  20.37  21.20    19.20

[101 rows x 5 columns]

Highest source concentration by agent:
changed_by
AGT0000007    24.82
AGT0000009    24.78
AGT0000080    24.48
AGT0000027    24.46
AGT0000017    24.44
AGT0000002    23.92
AGT0000073    23.61
AGT0000081    23.58
AGT0000044    23.52
AGT0000043    23.39
AGT0000097    23.37
AGT0000004    23.30
AGT0000058    23.30
AGT

In [41]:
# Check for agents with unusually high activity

agent_activity = history["changed_by"].value_counts()

print("Agent activity statistics:")
print("Average:", round(agent_activity.mean(), 2))
print("Median:", agent_activity.median())
print("Standard deviation:", round(agent_activity.std(), 2))

print("\nAgents with unusually high activity:")
print(agent_activity[agent_activity > agent_activity.mean() + 2 * agent_activity.std()])


Agent activity statistics:
Average: 594.06
Median: 594.0
Standard deviation: 25.34

Agents with unusually high activity:
Series([], Name: count, dtype: int64)


In [42]:
# Check for agents with unusually low activity

low_activity = agent_activity[
    agent_activity < agent_activity.mean() - 2 * agent_activity.std()
]

print("Agents with unusually low activity:")
print(low_activity)

Agents with unusually low activity:
changed_by
AGT0000015    542
AGT0000064    537
AGT0000049    527
Name: count, dtype: int64


In [43]:
# Check agent activity consistency across sources

agent_source_counts = pd.crosstab(
    history["changed_by"],
    history["source"]
)

print("Agent activity consistency across sources:")
print(agent_source_counts.round(2))

print("\nAgents with highest source variation:")

source_std = agent_source_counts.std(axis=1).sort_values(ascending=False)

print(source_std.head(10))

Agent activity consistency across sources:
source      BATCH  CALL  CORE  FIELD  PAYMENT
changed_by                                   
AGT0000001    120   130   134    126      123
AGT0000002    106   138   110     98      125
AGT0000003    134   135   116    120      117
AGT0000004    128   147   115    119      122
AGT0000005    137   111   122    107      128
...           ...   ...   ...    ...      ...
AGT0000097    118   103   136     95      130
AGT0000098    127   117   134    131      122
AGT0000099    110   122   109    126      123
AGT0000100    112   108   118    109      108
SYSTEM        101   134   122    127      115

[101 rows x 5 columns]

Agents with highest source variation:
changed_by
AGT0000009    20.777392
AGT0000017    19.690099
AGT0000037    19.659603
AGT0000060    19.217180
AGT0000007    18.061008
AGT0000097    17.386777
AGT0000065    16.902663
AGT0000002    15.993749
AGT0000043    15.773395
AGT0000081    15.562776
dtype: float64


In [44]:
# Check for agents heavily concentrated in one source

agent_source_percentage = (
    agent_source_counts.div(
        agent_source_counts.sum(axis=1),
        axis=0
    ) * 100
)

max_source_percentage = agent_source_percentage.max(axis=1)

print("Highest source concentration for each agent:")
print(max_source_percentage.sort_values(ascending=False).head(15))

print("\nAgents with source concentration above 30%:")
print(max_source_percentage[max_source_percentage > 30])

Highest source concentration for each agent:
changed_by
AGT0000007    24.816176
AGT0000009    24.784854
AGT0000080    24.475524
AGT0000027    24.459235
AGT0000017    24.440620
AGT0000002    23.916811
AGT0000073    23.611111
AGT0000081    23.580034
AGT0000044    23.519164
AGT0000043    23.385689
AGT0000097    23.367698
AGT0000004    23.296355
AGT0000058    23.296355
AGT0000065    23.063973
AGT0000078    23.051410
dtype: float64

Agents with source concentration above 30%:
Series([], dtype: float64)


In [45]:
# Check for suspicious SYSTEM activity

system_activity = history[history["changed_by"] == "SYSTEM"]

print("SYSTEM status changes:", len(system_activity))

print("\nSYSTEM activity by source:")
print(system_activity["source"].value_counts())

print("\nSYSTEM activity by status:")
print(system_activity["status"].value_counts())

SYSTEM status changes: 599

SYSTEM activity by source:
source
CALL       134
FIELD      127
CORE       122
PAYMENT    115
BATCH      101
Name: count, dtype: int64

SYSTEM activity by status:
status
CLOSED        95
PAID          91
WRITEOFF      90
NPA           87
DELINQUENT    85
ACTIVE        76
PTP           75
Name: count, dtype: int64


In [46]:
# Check SYSTEM activity over time

system_monthly = (
    system_activity
    .assign(month=system_activity["event_at"].dt.to_period("M"))
    .groupby("month")
    .size()
)

print("Monthly SYSTEM activity:")
print(system_monthly)

print("\nHighest SYSTEM activity months:")
print(system_monthly.sort_values(ascending=False).head(5))

Monthly SYSTEM activity:
month
2026-01    92
2026-02    62
2026-03    90
2026-04    94
2026-05    80
2026-06    73
2026-07    82
2026-08    26
Freq: M, dtype: int64

Highest SYSTEM activity months:
month
2026-04    94
2026-01    92
2026-03    90
2026-07    82
2026-05    80
Freq: M, dtype: int64


In [47]:
# Check whether SYSTEM changes are unusually concentrated in one status

system_status_pct = (
    system_activity["status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("SYSTEM status percentage:")
print(system_status_pct)

print("\nMost common SYSTEM status:")
print(system_status_pct.head(3))

SYSTEM status percentage:
status
CLOSED        15.86
PAID          15.19
WRITEOFF      15.03
NPA           14.52
DELINQUENT    14.19
ACTIVE        12.69
PTP           12.52
Name: proportion, dtype: float64

Most common SYSTEM status:
status
CLOSED      15.86
PAID        15.19
WRITEOFF    15.03
Name: proportion, dtype: float64


In [48]:
# Check rapid SYSTEM status changes

system_sorted = system_activity.sort_values(["account_id", "event_at"]).copy()

system_sorted["previous_event_at"] = (
    system_sorted.groupby("account_id")["event_at"].shift(1)
)

system_sorted["hours_since_previous"] = (
    system_sorted["event_at"] - system_sorted["previous_event_at"]
).dt.total_seconds() / 3600

system_rapid = system_sorted[
    system_sorted["hours_since_previous"].notna()
    & (system_sorted["hours_since_previous"] <= 24)
]

print("SYSTEM changes within 24 hours:", len(system_rapid))

print("\nSYSTEM changes within 1 hour:")
print(
    system_sorted[
        system_sorted["hours_since_previous"].notna()
        & (system_sorted["hours_since_previous"] <= 1)
    ][
        ["account_id", "previous_event_at", "event_at",
         "hours_since_previous", "status", "source"]
    ].head(15)
)


SYSTEM changes within 24 hours: 1

SYSTEM changes within 1 hour:
Empty DataFrame
Columns: [account_id, previous_event_at, event_at, hours_since_previous, status, source]
Index: []


In [49]:
# Inspect call timestamps

calls = data["calls"].copy()

print(calls["event_at"].head(10))
print("\nTimestamp data type:")
print(calls["event_at"].dtype)

0    2026-07-15 15:36:22
1    2026-06-10 06:48:27
2    2026-04-07 00:35:35
3    2026-02-12 14:16:57
4    2026-05-24 15:33:12
5    2026-02-04 11:37:08
6    2026-02-10 10:59:05
7    2026-06-14 23:37:56
8    2026-03-17 16:24:42
9    2026-06-24 04:26:34
Name: event_at, dtype: str

Timestamp data type:
str


In [50]:
# Inspect call timestamp patterns

calls["event_at"] = pd.to_datetime(calls["event_at"])

print("Earliest call:", calls["event_at"].min())
print("Latest call:", calls["event_at"].max())

print("\nCalls by hour:")
print(calls["event_at"].dt.hour.value_counts().sort_index())


Earliest call: 2025-12-29 06:52:37
Latest call: 2026-08-12 15:43:05

Calls by hour:
event_at
0     3633
1     3747
2     3884
3     3777
4     3700
5     3800
6     3819
7     3797
8     3704
9     3702
10    3795
11    3851
12    3680
13    3813
14    3702
15    3645
16    3843
17    3637
18    3788
19    3778
20    3684
21    3717
22    3685
23    3898
Name: count, dtype: int64


In [51]:
# Check vendor telephony data

vendor = data["vendor_telephony"].copy()

print(vendor.columns.tolist())
print("\nRows:", len(vendor))
vendor.head()

['vendor_id', 'vendor_name', 'vendor_account_id', 'timezone', 'status', 'schema_version']

Rows: 15


,vendor_id,vendor_name,vendor_account_id,timezone,status,schema_version
0,VND0000001,Airtel,VAC342762,Asia/Kolkata,INACTIVE,v3
1,VND0000002,Exotel,VAC456766,UTC,INACTIVE,v3
2,VND0000003,Twilio,VAC074211,UTC,INACTIVE,v3
3,VND0000004,Twilio,VAC321507,UTC,ACTIVE,v1
4,VND0000005,Twilio,VAC976733,Asia/Kolkata,ACTIVE,v1


In [52]:
# Check vendor usage in calls

vendor_usage = calls["vendor_id"].value_counts().reset_index()
vendor_usage.columns = ["vendor_id", "call_count"]

vendor_usage = vendor_usage.merge(
    vendor[["vendor_id", "vendor_name", "timezone", "status", "schema_version"]],
    on="vendor_id",
    how="left"
)

vendor_usage

,vendor_id,call_count,vendor_name,timezone,status,schema_version
0,VND0000010,6167,Airtel,UTC,INACTIVE,v1
1,VND0000011,6078,Airtel,UTC,ACTIVE,v3
2,VND0000007,6074,TataTele,UTC,INACTIVE,v1
3,VND0000015,6071,TataTele,Asia/Kolkata,ACTIVE,v2
4,VND0000005,6055,Twilio,Asia/Kolkata,ACTIVE,v1
5,VND0000014,6046,Knowlarity,Asia/Kolkata,INACTIVE,v3
6,VND0000001,6031,Airtel,Asia/Kolkata,INACTIVE,v3
7,VND0000012,6015,Airtel,UTC,INACTIVE,v1
8,VND0000009,5976,Exotel,Asia/Kolkata,INACTIVE,v1
9,VND0000003,5966,Twilio,UTC,INACTIVE,v3


In [53]:
# Check timezone distribution

timezone_usage = (
    vendor_usage.groupby("timezone")["call_count"]
    .agg(["count", "sum"])
    .reset_index()
)

timezone_usage

,timezone,count,sum
0,Asia/Kolkata,7,42005
1,UTC,8,48074


In [54]:
# Check vendor mapping

vendor[["vendor_id", "vendor_name", "timezone", "schema_version", "status"]].sort_values("vendor_id")

,vendor_id,vendor_name,timezone,schema_version,status
0,VND0000001,Airtel,Asia/Kolkata,v3,INACTIVE
1,VND0000002,Exotel,UTC,v3,INACTIVE
2,VND0000003,Twilio,UTC,v3,INACTIVE
3,VND0000004,Twilio,UTC,v1,ACTIVE
4,VND0000005,Twilio,Asia/Kolkata,v1,ACTIVE
5,VND0000006,Knowlarity,UTC,v2,ACTIVE
6,VND0000007,TataTele,UTC,v1,INACTIVE
7,VND0000008,Airtel,Asia/Kolkata,v3,INACTIVE
8,VND0000009,Exotel,Asia/Kolkata,v1,INACTIVE
9,VND0000010,Airtel,UTC,v1,INACTIVE


In [56]:

print(calls_timezone.columns.tolist())

['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone_x', 'timezone_y']


In [57]:
# Check vendor timezone mapping

print("Missing vendor timezone:", calls_timezone["timezone_y"].isna().sum())

calls_timezone[["vendor_id", "event_at", "timezone_x", "timezone_y"]].head(10)

Missing vendor timezone: 0


,vendor_id,event_at,timezone_x,timezone_y
0,VND0000013,2026-07-15 15:36:22,Asia/Dubai,Asia/Kolkata
1,VND0000001,2026-06-10 06:48:27,Asia/Kolkata,Asia/Kolkata
2,VND0000008,2026-04-07 00:35:35,Asia/Dubai,Asia/Kolkata
3,VND0000001,2026-02-12 14:16:57,Asia/Dubai,Asia/Kolkata
4,VND0000006,2026-05-24 15:33:12,Asia/Kolkata,UTC
5,VND0000011,2026-02-04 11:37:08,Asia/Dubai,UTC
6,VND0000009,2026-02-10 10:59:05,Asia/Kolkata,Asia/Kolkata
7,VND0000003,2026-06-14 23:37:56,Asia/Dubai,UTC
8,VND0000012,2026-03-17 16:24:42,UTC,UTC
9,VND0000014,2026-06-24 04:26:34,UTC,Asia/Kolkata


In [58]:
# Find timezone mismatches

calls_timezone["timezone_match"] = (
    calls_timezone["timezone_x"] == calls_timezone["timezone_y"]
)

timezone_check = (
    calls_timezone["timezone_match"]
    .value_counts()
    .rename_axis("timezone_match")
    .reset_index(name="call_count")
)

timezone_check["percentage"] = (
    timezone_check["call_count"] / len(calls_timezone) * 100
).round(2)

timezone_check

,timezone_match,call_count,percentage
0,False,60078,66.69
1,True,30001,33.31


In [59]:
# Check timezone mismatch patterns

timezone_mismatch = (
    calls_timezone[~calls_timezone["timezone_match"]]
    .groupby(["timezone_x", "timezone_y"])
    .size()
    .reset_index(name="call_count")
    .sort_values("call_count", ascending=False)
)

timezone_mismatch["percentage"] = (
    timezone_mismatch["call_count"] / len(calls_timezone) * 100
).round(2)

timezone_mismatch

,timezone_x,timezone_y,call_count,percentage
2,Asia/Kolkata,UTC,16068,17.84
1,Asia/Dubai,UTC,16004,17.77
0,Asia/Dubai,Asia/Kolkata,14015,15.56
3,UTC,Asia/Kolkata,13991,15.53


In [60]:
# Check vendor account mapping

vendor_mapping = vendor[
    ["vendor_id", "vendor_name", "vendor_account_id", "timezone", "schema_version"]
].sort_values("vendor_name")

vendor_mapping


,vendor_id,vendor_name,vendor_account_id,timezone,schema_version
0,VND0000001,Airtel,VAC342762,Asia/Kolkata,v3
7,VND0000008,Airtel,VAC496672,Asia/Kolkata,v3
9,VND0000010,Airtel,VAC172916,UTC,v1
10,VND0000011,Airtel,VAC461665,UTC,v3
11,VND0000012,Airtel,VAC583104,UTC,v1
1,VND0000002,Exotel,VAC456766,UTC,v3
8,VND0000009,Exotel,VAC382568,Asia/Kolkata,v1
5,VND0000006,Knowlarity,VAC544470,UTC,v2
12,VND0000013,Knowlarity,VAC138790,Asia/Kolkata,v1
13,VND0000014,Knowlarity,VAC547425,Asia/Kolkata,v3


In [61]:
# Check vendor usage by month

calls_timezone["month"] = calls_timezone["event_at"].dt.to_period("M")

vendor_monthly = pd.crosstab(
    calls_timezone["month"],
    calls_timezone["vendor_id"]
)

vendor_monthly

vendor_id,VND0000001,VND0000002,VND0000003,VND0000004,VND0000005,VND0000006,VND0000007,VND0000008,VND0000009,VND0000010,VND0000011,VND0000012,VND0000013,VND0000014,VND0000015
month,,,,,,,,,,,,,,,
2025-12,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2026-01,881,818,867,849,849,805,885,855,833,889,856,792,808,849,862
2026-02,749,795,772,735,821,772,768,774,794,779,731,778,739,769,804
2026-03,843,868,865,846,865,841,845,837,855,834,875,922,839,819,903
2026-04,823,806,812,792,805,797,828,776,834,874,840,808,846,826,791
2026-05,898,844,842,837,846,867,855,834,809,872,823,826,819,893,889
2026-06,770,792,803,825,834,772,813,822,801,827,852,859,788,784,795
2026-07,840,819,783,829,817,833,864,797,868,870,871,827,859,879,827
2026-08,227,199,222,217,218,216,216,222,182,222,230,203,210,227,200


In [62]:
# Check agent identity mapping

agents = data["agents"].copy()

print("Total agent records:", len(agents))
print("Unique agent IDs:", agents["agent_id"].nunique())
print("Unique agent names:", agents["agent_name"].nunique())

Total agent records: 30000
Unique agent IDs: 1000
Unique agent names: 10


In [63]:
# Check agent IDs per name

agent_name_check = (
    agents.groupby("agent_name")["agent_id"]
    .nunique()
    .reset_index(name="unique_agent_ids")
    .sort_values("unique_agent_ids", ascending=False)
)

agent_name_check

,agent_name,unique_agent_ids
8,Sneha Das,958
5,Priya Mehta,957
4,Pooja Nair,954
1,Amit Kumar,952
3,Neha Singh,951
0,Aarav Sharma,946
2,Ananya Rao,946
7,Rohan Patel,946
6,Rahul Verma,943
9,Vikram Shah,936


In [64]:
# Calculate monthly recovery

payments = data["payments"].copy()
payments["event_at"] = pd.to_datetime(payments["event_at"])

monthly_recovery = (
    payments[payments["payment_status"] == "SUCCESS"]
    .groupby(payments["event_at"].dt.to_period("M"))["amount"]
    .sum()
    .reset_index()
)

monthly_recovery.columns = ["month", "recovery_amount"]

monthly_recovery


,month,recovery_amount
0,2026-01,1.872291e+08
1,2026-02,1.702796e+08
2,2026-03,1.891903e+08
3,2026-04,1.752289e+08
4,2026-05,1.843355e+08
5,2026-06,1.758534e+08
6,2026-07,1.872478e+08
7,2026-08,4.710970e+07


In [65]:
# Calculate month-on-month growth

monthly_recovery["mom_growth"] = (
    monthly_recovery["recovery_amount"].pct_change() * 100
).round(2)

monthly_recovery

,month,recovery_amount,mom_growth
0,2026-01,1.872291e+08,NaN
1,2026-02,1.702796e+08,-9.05
2,2026-03,1.891903e+08,11.11
3,2026-04,1.752289e+08,-7.38
4,2026-05,1.843355e+08,5.20
5,2026-06,1.758534e+08,-4.60
6,2026-07,1.872478e+08,6.48
7,2026-08,4.710970e+07,-74.84


In [66]:
# Check monthly account base

accounts = data["accounts"].copy()
accounts["opened_at"] = pd.to_datetime(accounts["opened_at"])

monthly_accounts = (
    accounts.groupby(accounts["opened_at"].dt.to_period("M"))
    .size()
    .reset_index(name="accounts_opened")
)

monthly_accounts

,opened_at,accounts_opened
0,2024-01,1311
1,2024-02,1299
2,2024-03,1404
3,2024-04,1263
4,2024-05,1287
5,2024-06,1219
6,2024-07,1324
7,2024-08,1358
8,2024-09,1326
9,2024-10,1362


In [67]:
# Check daily targeting data

targeting = data["daily_targeting"].copy()

print(targeting.columns.tolist())
print("\nRows:", len(targeting))
targeting.head()

['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']

Rows: 45000


,target_id,account_id,campaign_id,target_date,priority,recommended_channel,status
0,TGT0000001,ACC0028555,CMP0000103,2026-04-22,4,FIELD,QUEUED
1,TGT0000002,ACC0007194,CMP0000100,2026-08-06,10,SMS,EXPIRED
2,TGT0000003,ACC0029550,CMP0000074,2026-05-20,5,SMS,CONTACTED
3,TGT0000004,ACC0012329,CMP0000046,2026-07-30,10,WHATSAPP,QUEUED
4,TGT0000005,ACC0018387,CMP0000118,2026-07-10,7,WHATSAPP,EXPIRED


In [68]:
# Check monthly targeting volume

targeting["target_date"] = pd.to_datetime(targeting["target_date"])

monthly_targeting = (
    targeting.groupby(targeting["target_date"].dt.to_period("M"))
    .size()
    .reset_index(name="targeted_accounts")
)

monthly_targeting

,target_date,targeted_accounts
0,2026-01,6369
1,2026-02,5709
2,2026-03,6290
3,2026-04,6205
4,2026-05,6442
5,2026-06,6154
6,2026-07,6230
7,2026-08,1601


In [69]:
# Calculate monthly recovery rate

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

successful_payments["month"] = (
    successful_payments["event_at"].dt.to_period("M")
)

monthly_paid_accounts = (
    successful_payments.groupby("month")["account_id"]
    .nunique()
    .reset_index(name="paid_accounts")
)

monthly_recovery_rate = monthly_targeting.merge(
    monthly_paid_accounts,
    left_on="target_date",
    right_on="month",
    how="left"
)

monthly_recovery_rate["recovery_rate"] = (
    monthly_recovery_rate["paid_accounts"]
    / monthly_recovery_rate["targeted_accounts"]
    * 100
).round(2)

monthly_recovery_rate


,target_date,targeted_accounts,month,paid_accounts,recovery_rate
0,2026-01,6369,2026-01,2374,37.27
1,2026-02,5709,2026-02,2173,38.06
2,2026-03,6290,2026-03,2419,38.46
3,2026-04,6205,2026-04,2304,37.13
4,2026-05,6442,2026-05,2344,36.39
5,2026-06,6154,2026-06,2286,37.15
6,2026-07,6230,2026-07,2335,37.48
7,2026-08,1601,2026-08,608,37.98
